In [2]:
import requests
import pandas as pd
import zipfile
import io
import duckdb

In [34]:
url_sau = "https://www.data.gouv.fr/api/1/datasets/r/b27d31a6-107b-46ee-8427-518799b488f0"
df_sau = pd.read_csv(url_sau, sep=',')
df_sau

,date_mesure,geocode_commune,libelle_commune,valeur
0,2020-01-01T00:00:00.000,49323,Verrières-en-Anjou,852.10
1,2020-01-01T00:00:00.000,48153,Saint-Gal,279.44
2,2010-01-01T00:00:00.000,25570,Tressandans,128.87
3,2010-01-01T00:00:00.000,28194,Houville-la-Branche,1528.92
4,2010-01-01T00:00:00.000,50294,Martinvast,545.78
...,...,...,...,...
67545,2020-01-01T00:00:00.000,91662,Villeconin,980.82
67546,2020-01-01T00:00:00.000,54270,Hussigny-Godbrange,136.16
67547,2020-01-01T00:00:00.000,61238,Louvières-en-Auge,693.46
67548,2020-01-01T00:00:00.000,63274,Perpezat,3435.95


In [37]:
df_sau = df_sau[df_sau['date_mesure'].str.startswith('2020', na=False)].copy()
df_sau['geocode_commune']= df_sau['geocode_commune'].str.zfill(5)
df_sau

,date_mesure,geocode_commune,libelle_commune,valeur
0,2020-01-01T00:00:00.000,49323,Verrières-en-Anjou,852.10
1,2020-01-01T00:00:00.000,48153,Saint-Gal,279.44
11,2020-01-01T00:00:00.000,09109,Durfort,1021.46
12,2020-01-01T00:00:00.000,30155,Manduel,1083.21
13,2020-01-01T00:00:00.000,12045,Camboulazet,921.58
...,...,...,...,...
67545,2020-01-01T00:00:00.000,91662,Villeconin,980.82
67546,2020-01-01T00:00:00.000,54270,Hussigny-Godbrange,136.16
67547,2020-01-01T00:00:00.000,61238,Louvières-en-Auge,693.46
67548,2020-01-01T00:00:00.000,63274,Perpezat,3435.95


In [ ]:
URL = (
        "https://www.insee.fr/fr/statistiques/fichier/4505239/ODD_PARQUET.zip"
    )
    zip_content = download_file(URL)
    with zipfile.ZipFile(BytesIO(zip_content)) as z:
        with z.open("catnat_gaspar.csv") as f:
            df_cat_nat = pd.read_csv(f, sep=";", low_memory=False)

In [11]:
#Chargement local
df_communes = duckdb.read_parquet("../data/data_sau/raw/ODD_COM.parquet")

In [6]:
df_communes

┌───────────┬─────────────────────────────────────────────────────────────────────┬──────────┬─────────┬─────────┬─────────┬─────────┬──────────┬───────────────┬────────────┬────────┬────────┬────────┬────────┬────────────────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│  codgeo   │                               libgeo                                │ no_indic │  ODD1   │ Cible1  │  ODD2   │ Cible2  │ type_var │   variable    │ sous_champ │ A2025  │ A2024  │ A2023  │ A2022  │       A2021        │ A2020  │ A2019  │ A2018  │ A2017  │ A2016  │ A2015  │ A2014  │ A2013  │ A2012  │ A2011  │ A2010  │ A2009  │ A2008  │ A2007  │ A2006  │ A2005  │ A2004  │ A2003  │ A2002  │ A2001  │ A2000

In [ ]:
query = """ 
SELECT 
    codgeo,
    libgeo,
    A2021 AS surface
FROM df_communes
WHERE variable = 'surface'
"""

df_surf_com = duckdb.sql(query)

In [32]:
df_epci.columns

Index(['code_insee', 'nom', 'pop_tot_commune', 'pop_mun_commune', 'siren',
       'epci_nom', 'epci_type', 'epci_modeFinancement', 'total_pop_tot',
       'total_pop_mun', 'superficie_hectare', 'superficie_km2', 'dept_com',
       'bassin_vie', 'dept_epci'],
      dtype='object')

In [40]:
query = """
SELECT
    dept_epci as dept_id,
    siren as id_epci,
    epci_nom AS lib_epci,
    'i113' AS id_indicator,
    ROUND(sum(df_sau.valeur/100) / sum(surface)  * 100,3) AS valeur_brute,
    '2020' AS annee
FROM df_epci
LEFT JOIN df_surf_com 
    ON df_epci.code_insee = df_surf_com.codgeo
LEFT JOIN df_sau
    ON df_epci.code_insee = df_sau.geocode_commune
GROUP BY siren, dept_epci, epci_nom
ORDER BY valeur_brute DESC
"""

df_sau_final = duckdb.sql(query)

In [41]:
df_sau_final

┌─────────┬───────────┬───────────────────────────────────┬──────────────┬──────────────┬─────────┐
│ dept_id │  id_epci  │             lib_epci              │ id_indicator │ valeur_brute │  annee  │
│ varchar │  varchar  │              varchar              │   varchar    │    double    │ varchar │
├─────────┼───────────┼───────────────────────────────────┼──────────────┼──────────────┼─────────┤
│ 80      │ 200070928 │ CC Terre de Picardie              │ i113         │       94.599 │ 2020    │
│ 28      │ 200070159 │ CC Coeur de Beauce                │ i113         │       94.142 │ 2020    │
│ 45      │ 244500542 │ CC de la Plaine du Nord Loiret    │ i113         │       93.412 │ 2020    │
│ 67      │ 200034635 │ CC du Kochersberg                 │ i113         │       93.032 │ 2020    │
│ 76      │ 247600505 │ CC Campagne-de-Caux               │ i113         │       91.773 │ 2020    │
│ 02      │ 240200634 │ CC des Portes de la Thiérache     │ i113         │       91.303 │ 2020    │


In [30]:
duckdb.sql("""SELECT * FROM df_surf_epci WHERE siren = '200071553'""")

┌───────────┬───────────┬─────────┐
│ dept_epci │   siren   │ surface │
│  varchar  │  varchar  │ double  │
├───────────┼───────────┼─────────┤
│ 49        │ 200071553 │  613.55 │
└───────────┴───────────┴─────────┘

In [9]:
df_epci = pd.read_csv("../data/processed/epci_membres.csv", sep=",", dtype=str)

In [31]:
sum(df_epci[df_epci['siren'] == "200071553"]['superficie_km2'].astype(float))

353.0

In [10]:
df_sau = df_sau[df_sau['date_mesure'].str.startswith('2020')]
df_sau

,date_mesure,geocode_epci,libelle_epci,valeur
1,2020-01-01T00:00:00.000,247400773,Communauté de communes des Sources du Lac d'An...,2228.34
2,2020-01-01T00:00:00.000,245501176,Communauté de communes du Territoire de Fresne...,15404.11
7,2020-01-01T00:00:00.000,200066785,Communauté de communes de l'Oust à Brocéliande,37332.64
10,2020-01-01T00:00:00.000,200040574,Communauté de communes Beaujolais Pierres Dorées,8968.44
12,2020-01-01T00:00:00.000,242010130,Communauté de communes du Sartenais Valinco Ta...,9759.43
...,...,...,...,...
2499,2020-01-01T00:00:00.000,240800862,Communauté de communes des Crêtes Préardennaises,62670.81
2500,2020-01-01T00:00:00.000,244301131,Communauté de communes Loire et Semène,4687.10
2501,2020-01-01T00:00:00.000,249500489,Communauté de communes du Haut Val d'Oise,1769.72
2502,2020-01-01T00:00:00.000,246700967,Communauté de communes de Sélestat,8109.07
